In [1]:
import pandas as pd
import plotly.express as px

In [2]:
cometa_df = pd.read_excel('Archivos/C_2006_1P.xlsx', sheet_name='Hoja3')
cometa_df

,Date (UT),Magn
0,7C2002 1 13.07265,20.5 P
1,7C2002 1 13.07348,P
2,7C2002 1 13.07597,P
3,7C2005 1 18.45625,19 P
4,7C2005 1 18.45707,P
...,...,...
4260,C2013 1 6.3737,16.6 P
4261,C2013 1 6.37912,16.7 P
4262,C2013 1 6.39325,16.7 P
4263,`C2012 12 29.72922,18.6 P


In [3]:
filas,columnas = cometa_df.shape
print(f'Registros: {filas}\nVariables: {columnas}')

Registros: 4265
Variables: 2


In [4]:
cometa_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4265 entries, 0 to 4264
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   Date (UT)  4265 non-null   object
 1   Magn       4265 non-null   object
dtypes: object(2)
memory usage: 66.8+ KB


In [5]:
cometa_df.columns = cometa_df.columns.str.lower().str.replace(' ', '_').str.replace('(', '').str.replace(')', '')
cometa_df.sample(10) 

,date_ut,magn
3933,C2012 12 10.72959,15.9 P
888,KC2012 12 29.70481,16.6 P
3021,|C2012 11 20.30278,P
1120,C2012 9 14.9646,14.9 P
4190,C2012 12 7.84911,18.6 P
2559,C2012 12 30.83777,15.3 P
1595,C2012 9 24.73792,14 P
3759,KC2013 1 4.0805,18.8 P
467,C2012 12 30.025,15.7 P
2047,C2012 12 31.61656,12.1 P


In [6]:
cometa_df.magn = cometa_df.magn.str.extract(r'(\d+\.\d+)')
cometa_df.sample(10)

,date_ut,magn
3888,C2012 11 29.68774,15.4
527,C2013 1 5.38253,15.1
92,KC2013 1 3.55995,17.1
2161,C2013 1 3.71081,12.2
513,C2013 1 4.06832,15.8
1631,C2012 9 28.7418,14.3
284,C2012 12 30.82058,14.5
1643,C2012 9 29.734,14.4
3613,KC2012 12 25.99205,16.9
1356,C2012 12 30.04973,14.6


In [7]:
cometa_df['magn'] = pd.to_numeric(cometa_df.magn)
cometa_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4265 entries, 0 to 4264
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   date_ut  4265 non-null   object 
 1   magn     3655 non-null   float64
dtypes: float64(1), object(1)
memory usage: 66.8+ KB


In [8]:
cometa_filtrado_df = cometa_df #[cometa_df.magn > -60].copy()
cometa_filtrado_df

,date_ut,magn
0,7C2002 1 13.07265,20.5
1,7C2002 1 13.07348,NaN
2,7C2002 1 13.07597,NaN
3,7C2005 1 18.45625,NaN
4,7C2005 1 18.45707,NaN
...,...,...
4260,C2013 1 6.3737,16.6
4261,C2013 1 6.37912,16.7
4262,C2013 1 6.39325,16.7
4263,`C2012 12 29.72922,18.6


In [9]:
cometa_dias_df = cometa_filtrado_df.date_ut.str.extract(r'(\d+\.\d+)')
cometa_dias_df = pd.to_numeric(cometa_dias_df[0])

cometa_procesado_df = pd.DataFrame()
cometa_procesado_df['obs_date'] = cometa_filtrado_df.date_ut.str.extract(r'(\d+ \d+)') 
cometa_procesado_df.obs_date = cometa_procesado_df.obs_date.str.replace(' ', '-')
cometa_procesado_df.obs_date = cometa_procesado_df.obs_date  + '-' + cometa_dias_df.apply(lambda dato: str(int(dato)))
cometa_procesado_df.obs_date = pd.to_datetime(pd.to_datetime(cometa_procesado_df.obs_date).dt.date)

cometa_procesado_df['magnitude'] = cometa_filtrado_df.magn
cometa_procesado_df.reset_index(inplace = True, drop= True)

cometa_procesado_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4265 entries, 0 to 4264
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   obs_date   4265 non-null   datetime64[ns]
 1   magnitude  3655 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 66.8 KB


In [10]:
cometa_procesado_df

,obs_date,magnitude
0,2002-01-13,20.5
1,2002-01-13,NaN
2,2002-01-13,NaN
3,2005-01-18,NaN
4,2005-01-18,NaN
...,...,...
4260,2013-01-06,16.6
4261,2013-01-06,16.7
4262,2013-01-06,16.7
4263,2012-12-29,18.6


In [11]:
fig = px.scatter(cometa_procesado_df, x='obs_date', y='magnitude', template= 'plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.show()

In [12]:
cometa_procesado_df.to_parquet('Archivos/12P_Procesado_MPC.parquet', index = False)

In [13]:
import requests
import pandas as pd

response = requests.get("https://data.minorplanetcenter.net/api/get-obs", json={"desigs": [r"2525"], "output_format":["OBS_DF"]})

if response.ok:
    mpc_obs_crude_df = pd.DataFrame(response.json()[0]['OBS_DF'])
    mpc_obs_crude_df

else:
    print("Error: ", response.status_code, response.content)

In [14]:
mpc_obs_proceed_df = pd.DataFrame()
mpc_obs_proceed_df['obs_date'] = mpc_obs_crude_df.obs80.str[15:19] + '-' + mpc_obs_crude_df.obs80.str[20:22] + '-' + mpc_obs_crude_df.obs80.str[23:25]
mpc_obs_proceed_df['magnitude'] = mpc_obs_crude_df.obs80.str[65:70].str.replace(' ', '')

mpc_obs_proceed_df.obs_date = pd.to_datetime(mpc_obs_proceed_df.obs_date)
mpc_obs_proceed_df.magnitude = pd.to_numeric(mpc_obs_proceed_df.magnitude)

mpc_obs_proceed_df


,obs_date,magnitude
0,1931-12-05,13.50
1,1931-12-05,NaN
2,1931-12-08,NaN
3,1936-09-13,12.90
4,1939-02-15,NaN
...,...,...
10036,2026-01-18,16.48
10037,2026-01-20,16.35
10038,2026-01-20,16.45
10039,2026-01-20,16.47


In [15]:
mpc_obs_proceed_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10041 entries, 0 to 10040
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   obs_date   10041 non-null  datetime64[ns]
 1   magnitude  9720 non-null   float64       
dtypes: datetime64[ns](1), float64(1)
memory usage: 157.0 KB


In [16]:
# mpc_obs_filtered_df.magnitude.plot(kind='box')
px.box(mpc_obs_proceed_df, y='magnitude', template= 'plotly_dark')

In [17]:
q1 = mpc_obs_proceed_df['magnitude'].quantile(0.25)
q3 = mpc_obs_proceed_df['magnitude'].quantile(0.75)

IQR = q3 - q1 

limite_inferior = q1 - 1.5 * IQR
limite_superior = q3 + 1.5 * IQR

# Filtrar los outliers
mpc_obs_filtered_df = mpc_obs_proceed_df[(mpc_obs_proceed_df['magnitude'] >= limite_inferior) & (mpc_obs_proceed_df['magnitude'] <= limite_superior)]

px.box(mpc_obs_filtered_df, y='magnitude', template= 'plotly_dark')

In [18]:
fig = px.scatter(mpc_obs_filtered_df, x='obs_date', y='magnitude', template= 'plotly_dark')
fig.update_yaxes(autorange="reversed")
fig.show()

In [19]:
mpc_obs_filtered_df.to_parquet('Archivos/2525_OStein_MPC.parquet', index = False)